# 🚀 Freefield — Google Colab Pro GPU Backend (Persistent GDrive + Auto-Shutdown)

Notebook ini menjalankan backend GPU **Freefield** (`ComfyUI` + `comfy-api-proxy` + `infra/gateway.py`) di Google Colab Pro:
- **Persistent Models & Config di Google Drive**: Semua model AI (`/content/drive/MyDrive/freefield/models`) dan token (`/content/drive/MyDrive/freefield/colab.env`) disimpan permanen di Google Drive. **Cukup isi token & download model 1 kali saja di awal!**
- **Static URL Tunnel (Ngrok Static Domain / Cloudflare Tunnel)**: Tidak perlu ganti-ganti `COMFY_BASE_URL` di `.env` laptop jika memakai Ngrok Static Domain.
- **Serverless-Style Idle Auto-Shutdown**: Jika tidak ada job/request selama `IDLE_SHUTDOWN_MINUTES` (default 10 menit), VM Colab otomatis memanggil `google.colab.runtime.unassign()` agar **Compute Units (CU) Colab Pro kamu langsung berhenti terpakai**.

### Cara Pakai Harian (Setelah Setup Pertama):
1. Klik menu **Runtime → Run all (`Ctrl+F9`)** dan izinkan akses Google Drive.
2. Di laptop lokal kamu, jalankan `npm run serve` lalu buka `http://127.0.0.1:3000`.

In [1]:
#@title 1. Mount Google Drive & Load/Simpan Konfigurasi Permanen { display-mode: "form" }
#@markdown Isi kolom di bawah ini **HANYA PADA PEMAKAIAN PERTAMA** (atau jika ingin mengubahnya). Nilai yang diisi akan otomatis disimpan ke `/content/drive/MyDrive/freefield/colab.env` sehingga pada sesi berikutnya kamu cukup langsung tekan **Run All (`Ctrl+F9`)** walaupun kolom di bawah kosong.

import os
import re
import subprocess
from pathlib import Path
from google.colab import drive

GDRIVE_FREEFIELD_DIR = "/content/drive/MyDrive/freefield" #@param {type:"string"}
FREEFIELD_REPO_URL = "https://github.com/RegiMuhammar/freefield.git" #@param {type:"string"}
FREEFIELD_TOKEN_INPUT = "" #@param {type:"string"}
HF_TOKEN_INPUT = "" #@param {type:"string"}
TUNNEL_PROVIDER = "ngrok" #@param ["ngrok", "cloudflare"]
NGROK_AUTHTOKEN_INPUT = "" #@param {type:"string"}
NGROK_DOMAIN_INPUT = "" #@param {type:"string"}

# 1. Mount Google Drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

BASE_DIR = Path(GDRIVE_FREEFIELD_DIR)
MODELS_DIR = BASE_DIR / "models"
ENV_FILE = BASE_DIR / "colab.env"
BASE_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Baca colab.env dari Google Drive jika sudah pernah disimpan
saved_env: dict[str, str] = {}
if ENV_FILE.exists():
    for line in ENV_FILE.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            saved_env[k.strip()] = v.strip()

def _get_secret(key: str, form_val: str) -> str:
    if form_val.strip():
        return form_val.strip()
    if saved_env.get(key):
        return saved_env[key]
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val:
            return val.strip()
    except Exception:
        pass
    return ""

FREEFIELD_TOKEN = _get_secret("FREEFIELD_TOKEN", FREEFIELD_TOKEN_INPUT)
HF_TOKEN = _get_secret("HF_TOKEN", HF_TOKEN_INPUT)
NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN", NGROK_AUTHTOKEN_INPUT)
NGROK_DOMAIN = _get_secret("NGROK_DOMAIN", NGROK_DOMAIN_INPUT)
TUNNEL_PROVIDER = TUNNEL_PROVIDER or saved_env.get("TUNNEL_PROVIDER", "ngrok")

if not FREEFIELD_TOKEN:
    import secrets
    FREEFIELD_TOKEN = secrets.token_urlsafe(32)
    print(f"🔑 Membuat FREEFIELD_TOKEN baru secara otomatis: {FREEFIELD_TOKEN}")

# 3. Simpan kembali ke /content/drive/MyDrive/freefield/colab.env agar sesi berikutnya otomatis
env_lines = [
    f"FREEFIELD_TOKEN={FREEFIELD_TOKEN}",
    f"HF_TOKEN={HF_TOKEN}",
    f"TUNNEL_PROVIDER={TUNNEL_PROVIDER}",
    f"NGROK_AUTHTOKEN={NGROK_AUTHTOKEN}",
    f"NGROK_DOMAIN={NGROK_DOMAIN}",
]
ENV_FILE.write_text("\n".join(env_lines) + "\n", encoding="utf-8")
os.environ["FREEFIELD_TOKEN"] = FREEFIELD_TOKEN
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# 4. Cek GPU yang didapat di Colab
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True)
gpu_line = gpu_info.stdout.strip() if gpu_info.returncode == 0 else "No GPU detected"
print(f"✅ Google Drive terhubung: {BASE_DIR}")
print(f"✅ Folder Models Permanen: {MODELS_DIR}")
print(f"✅ File Config Tersimpan : {ENV_FILE}")
print(f"🖥️ GPU Colab Aktif       : {gpu_line}")

Mounted at /content/drive
✅ Google Drive terhubung: /content/drive/MyDrive/freefield
✅ Folder Models Permanen: /content/drive/MyDrive/freefield/models
✅ File Config Tersimpan : /content/drive/MyDrive/freefield/colab.env
🖥️ GPU Colab Aktif       : NVIDIA A100-SXM4-80GB, 81920 MiB


In [4]:
#@title 2. Setup Freefield & ComfyUI (Cached di Google Drive) { display-mode: "form" }
#@markdown Pada sesi pertama, cell ini meng-install ComfyUI + custom nodes ke `/content/freefield-comfy` lalu menyimpan arsip cache-nya ke Google Drive (`comfyui-repos-cache.tar.zst`). Pada sesi berikutnya, ekstraksi dari Google Drive hanya butuh **~15 di bawah**!

import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/freefield")
COMFY_DIR = Path("/content/freefield-comfy")
CACHE_TAR = BASE_DIR / "comfyui-repos-cache.tar.zst"

# Memastikan utilitas zstd ter-install agar tar bisa melakukan kompresi zstd
if shutil.which("zstd") is None:
    print("📦 Menginstall utilitas zstd untuk kompresi cepat...")
    subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "zstd"], check=True, stdout=subprocess.DEVNULL)

# Pastikan repo freefield ada di /content/freefield (atau sync dari Google Drive jika kamu menaruh repo di GDrive)
GDRIVE_REPO = BASE_DIR / "repo"
if GDRIVE_REPO.exists() and (GDRIVE_REPO / "tools" / "local_backend.py").exists():
    print(f"📂 Menyalin kode Freefield dari {GDRIVE_REPO} ke {REPO_DIR}...")
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.copytree(GDRIVE_REPO, REPO_DIR, symlinks=True)
elif not (REPO_DIR / "tools" / "local_backend.py").exists():
    print(f"🌐 Cloning repo Freefield dari {FREEFIELD_REPO_URL}...")
    subprocess.run(["git", "clone", FREEFIELD_REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

# Restore cache git ComfyUI + custom_nodes dari Google Drive jika ada
if CACHE_TAR.exists() and not (COMFY_DIR / ".git").exists():
    print(f"⚡ Mengekstrak cache ComfyUI dari Google Drive ({CACHE_TAR.name})...")
    try:
        subprocess.run(["tar", "--zstd", "-xf", str(CACHE_TAR), "-C", "/content"], check=True)
    except subprocess.CalledProcessError:
        print("⚠️ Cache rusak atau korup, menghapus cache lama dan mengunduh ulang...")
        CACHE_TAR.unlink(missing_ok=True)

# Jalankan local_backend.py setup (akan lanjut cepat jika sudah di-restore dari cache)
subprocess.run([
    sys.executable, str(REPO_DIR / "tools" / "local_backend.py"),
    "setup", "--models", str(MODELS_DIR), str(COMFY_DIR)
], check=True)

# Simpan cache ComfyUI + custom_nodes ke Google Drive jika belum ada
if not CACHE_TAR.exists():
    print(f"💾 Menyimpan cache ComfyUI ke Google Drive ({CACHE_TAR}) agar sesi berikutnya instan...")
    subprocess.run(["tar", "--zstd", "-cf", str(CACHE_TAR), "-C", "/content", "freefield-comfy"], check=True)
    print("✅ Cache tersimpan!")

⚡ Mengekstrak cache ComfyUI dari Google Drive (comfyui-repos-cache.tar.zst)...
⚠️ Cache rusak atau korup, menghapus cache lama dan mengunduh ulang...
💾 Menyimpan cache ComfyUI ke Google Drive (/content/drive/MyDrive/freefield/comfyui-repos-cache.tar.zst) agar sesi berikutnya instan...
✅ Cache tersimpan!


In [5]:
#@title 3. Download & Auto-Repair Pack Model di Google Drive 5 TB { display-mode: "form" }
#@markdown Centang pack yang ingin dipastikan tersedia di `/content/drive/MyDrive/freefield/models`.
#@markdown - **Auto-Repair Corrupt Files**: Otomatis memeriksa semua file `.safetensors` di Google Drive (termasuk `wan_2.1_vae.safetensors` yang terpotong saat disk penuh tadi), menghapus file yang corrupt/incomplete, lalu mendownload ulang hanya file yang rusak/kurang!
#@markdown - **Anti Disk Full**: Membersihkan `/root/.cache/huggingface` setiap selesai 1 pack.

import shutil
import struct

DOWNLOAD_ALL_355GB = True #@param {type:"boolean"}
PACK_IMAGE_FLUX3_18GB = False #@param {type:"boolean"}
PACK_IMAGE_ZIMAGE_12GB = False #@param {type:"boolean"}
PACK_UPSCALE_5GB = False #@param {type:"boolean"}
PACK_VIDEO_LTX25_66GB = False #@param {type:"boolean"}
PACK_VIDEO_WAN_CAMERA_40GB = False #@param {type:"boolean"}
PACK_VIDEO_WAN_EFFECTS_45GB = False #@param {type:"boolean"}
PACK_VIDEO_WAN_MOTION_25GB = False #@param {type:"boolean"}
PACK_VIDEO_SCAIL_RECAST_29GB = False #@param {type:"boolean"}
PACK_IMAGE_QWEN_EDIT_32GB = False #@param {type:"boolean"}
PACK_IMAGE_KREA2_STYLES_24GB = False #@param {type:"boolean"}
PACK_AUDIO_MUSIC_SFX_29GB = False #@param {type:"boolean"}
PACK_VIDEO_H3_76GB = False #@param {type:"boolean"}
DRY_RUN_ONLY = False #@param {type:"boolean"}

def clean_colab_cache():
    for p in ["/root/.cache/huggingface", "/content/.cache", "/tmp/hf_cache", str(MODELS_DIR / ".incoming")]:
        shutil.rmtree(p, ignore_errors=True)

def is_safetensors_valid(path: Path) -> bool:
    try:
        size = path.stat().st_size
        if size < 1024:
            return False
        with open(path, "rb") as f:
            header_len = struct.unpack("<Q", f.read(8))[0]
            if header_len == 0 or header_len > 100_000_000 or (8 + header_len) > size:
                return False
        return True
    except Exception:
        return False

# 1. Periksa seluruh file .safetensors di Google Drive dan hapus yang corrupt/incomplete
print("🔍 Memeriksa integritas file model di Google Drive...")
removed_corrupt = 0
for sf in MODELS_DIR.rglob("*.safetensors"):
    if not is_safetensors_valid(sf):
        print(f"  🗑️ Menghapus file corrupt/incomplete: {sf.relative_to(MODELS_DIR)} ({sf.stat().st_size / (1024**2):.1f} MB)")
        sf.unlink(missing_ok=True)
        removed_corrupt += 1
if removed_corrupt == 0:
    print("  ✅ Semua file yang ada memiliki header safetensors yang valid!")
else:
    print(f"  🛠️ {removed_corrupt} file corrupt telah dihapus dan akan didownload ulang.")

clean_colab_cache()
print(f"🧹 Sisa disk lokal VM Colab sekarang: {shutil.disk_usage('/').free / (1024 ** 3):.1f} GB")

all_pack_names = [
    "image-flux3", "image", "upscale", "video-ltx25",
    "video-wan-camera", "video-wan-effects", "video-wan", "video-scail",
    "image-qwen-edit", "image-krea2", "image-moss", "audio", "video-h3"
]

if DOWNLOAD_ALL_355GB:
    pack_list = all_pack_names
else:
    pack_list = [
        name for name, enabled in [
            ("image-flux3", PACK_IMAGE_FLUX3_18GB),
            ("image", PACK_IMAGE_ZIMAGE_12GB),
            ("upscale", PACK_UPSCALE_5GB),
            ("video-ltx25", PACK_VIDEO_LTX25_66GB),
            ("video-wan-camera", PACK_VIDEO_WAN_CAMERA_40GB),
            ("video-wan-effects", PACK_VIDEO_WAN_EFFECTS_45GB),
            ("video-wan", PACK_VIDEO_WAN_MOTION_25GB),
            ("video-scail", PACK_VIDEO_SCAIL_RECAST_29GB),
            ("image-qwen-edit", PACK_IMAGE_QWEN_EDIT_32GB),
            ("image-krea2", PACK_IMAGE_KREA2_STYLES_24GB),
            ("audio", PACK_AUDIO_MUSIC_SFX_29GB),
            ("video-h3", PACK_VIDEO_H3_76GB),
        ] if enabled
    ]

if pack_list:
    dl_env = {**os.environ, "HF_HUB_DISABLE_XET": "1", "HF_HUB_CACHE": "/tmp/hf_cache"}
    for pack in pack_list:
        print(f"\n📦 Memeriksa / Download pack: {pack}...")
        cmd = [
            sys.executable, str(REPO_DIR / "tools" / "local_backend.py"),
            "download", "--models", str(MODELS_DIR), str(COMFY_DIR), pack
        ]
        if DRY_RUN_ONLY:
            cmd.append("--dry-run")
        subprocess.run(cmd, env=dl_env, check=False)
        clean_colab_cache()
    print(f"\n✅ Selesai! Sisa disk lokal VM Colab: {shutil.disk_usage('/').free / (1024 ** 3):.1f} GB")
else:
    print("⏭️ Melewati pengecekan download model (tidak ada pack yang dicentang).")

🧹 Cache lokal VM Colab dibersihkan! Sisa disk lokal Colab sekarang: 233.1 GB

📦 Memeriksa / Download pack: image-flux3...

📦 Memeriksa / Download pack: image...

📦 Memeriksa / Download pack: upscale...

📦 Memeriksa / Download pack: video-ltx25...

📦 Memeriksa / Download pack: video-wan-camera...

📦 Memeriksa / Download pack: video-wan-effects...

📦 Memeriksa / Download pack: video-wan...

📦 Memeriksa / Download pack: video-scail...

📦 Memeriksa / Download pack: image-qwen-edit...

📦 Memeriksa / Download pack: image-krea2...

📦 Memeriksa / Download pack: image-moss...

📦 Memeriksa / Download pack: audio...

📦 Memeriksa / Download pack: video-h3...

✅ Selesai! Sisa disk lokal VM Colab: 217.6 GB


In [7]:
#@title 4. Jalankan Tunnel + Freefield Backend + Serverless Auto-Shutdown { display-mode: "form" }
#@markdown Cell ini menyalakan Tunnel HTTPS (Ngrok Static Domain atau Cloudflare Tunnel) dan menjalankan backend Freefield.
#@markdown - **`IDLE_UNLOAD_MINUTES`**: Mengosongkan VRAM GPU otomatis jika tidak ada request selama N menit.
#@markdown - **`IDLE_SHUTDOWN_MINUTES`**: Mematikan VM Google Colab (`runtime.unassign()`) otomatis jika tidak ada request/job selama N menit agar **Compute Units Colab Pro tidak terbuang**.

import json
import re
import subprocess
import sys
import time
import urllib.request

IDLE_UNLOAD_MINUTES = 5 #@param {type:"number"}
IDLE_SHUTDOWN_MINUTES = 10 #@param {type:"number"}
EXTRA_COMFY_ARGS = "auto" #@param {type:"string"}

# 1. Deteksi flag ComfyUI optimal berdasarkan VRAM GPU Colab
if EXTRA_COMFY_ARGS.strip() == "auto":
    vram_out = subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True)
    vram_mb = int(vram_out.stdout.strip().splitlines()[0]) if vram_out.returncode == 0 and vram_out.stdout.strip() else 0
    if vram_mb <= 17000:
        comfy_flags = ["--lowvram"]
    elif vram_mb <= 26000:
        comfy_flags = ["--reserve-vram", "1"]
    else:
        comfy_flags = []
    print(f"⚙️ Auto GPU Profile ({vram_mb} MB VRAM) → ComfyUI flags: {comfy_flags or '(default)'}")
else:
    comfy_flags = EXTRA_COMFY_ARGS.split()

# 2. Siapkan Tunnel (Ngrok atau Cloudflare)
tunnel_proc = None
public_url = ""

if TUNNEL_PROVIDER == "ngrok":
    if not NGROK_AUTHTOKEN:
        raise SystemExit("❌ NGROK_AUTHTOKEN belum diisi! Isi di Cell 1 atau ubah TUNNEL_PROVIDER ke 'cloudflare'.")
    if subprocess.run(["which", "ngrok"], capture_output=True).returncode != 0:
        subprocess.run(["pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_AUTHTOKEN
    ngrok.kill()
    if NGROK_DOMAIN:
        clean_domain = NGROK_DOMAIN.replace("https://", "").replace("http://", "").strip("/")
        try:
            listener = ngrok.connect(8000, "http", hostname=clean_domain)
        except Exception:
            listener = ngrok.connect(8000, "http")
    else:
        listener = ngrok.connect(8000, "http")
    public_url = listener.public_url.replace("http://", "https://")
else:
    cf_bin = Path("/usr/local/bin/cloudflared")
    if not cf_bin.exists():
        subprocess.run([
            "wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", str(cf_bin)
        ], check=True)
        cf_bin.chmod(0o755)
    cf_log = Path("/tmp/cloudflared.log")
    cf_log.unlink(missing_ok=True)
    tunnel_proc = subprocess.Popen([
        str(cf_bin), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate", "--logfile", str(cf_log)
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if cf_log.exists():
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", cf_log.read_text(encoding="utf-8", errors="ignore"))
            if m:
                public_url = m.group(0)
                break
        time.sleep(1)
    if not public_url:
        raise RuntimeError("Gagal mendapatkan URL dari cloudflared")

print("\n" + "=" * 72)
print("🎉 TUNNEL AKTIF! Pastikan baris ini ada di file .env laptop kamu:")
print(f"COMFY_BASE_URL={public_url}")
print(f"FREEFIELD_TOKEN={FREEFIELD_TOKEN}")
print("=" * 72 + "\n", flush=True)

# 3. Jalankan Backend dengan Idle Watchdog
idle_marker = Path("/tmp/freefield_idle_shutdown")
idle_marker.unlink(missing_ok=True)

env = {
    **os.environ,
    "FREEFIELD_TOKEN": FREEFIELD_TOKEN,
    "FREEFIELD_IDLE_UNLOAD_SEC": str(int(IDLE_UNLOAD_MINUTES * 60)) if IDLE_UNLOAD_MINUTES > 0 else "0",
    "FREEFIELD_IDLE_SHUTDOWN_SEC": str(int(IDLE_SHUTDOWN_MINUTES * 60)) if IDLE_SHUTDOWN_MINUTES > 0 else "0",
    "FREEFIELD_IDLE_MARKER": str(idle_marker),
}

# PENTING: "--models", str(MODELS_DIR) ditaruh SEBELUM str(COMFY_DIR)
cmd = [
    sys.executable, str(REPO_DIR / "tools" / "local_backend.py"),
    "serve", "--models", str(MODELS_DIR), str(COMFY_DIR),
    "--", *comfy_flags
]

try:
    rc = subprocess.call(cmd, env=env)
except KeyboardInterrupt:
    rc = 130
finally:
    if tunnel_proc and tunnel_proc.poll() is None:
        tunnel_proc.terminate()

# 4. Jika berhenti karena Idle Timeout, matikan Runtime VM Colab agar Compute Units berhenti terpakai!
if idle_marker.exists() or (rc == 0 and IDLE_SHUTDOWN_MINUTES > 0):
    print(f"\n💤 Idle selama {IDLE_SHUTDOWN_MINUTES} menit tanpa aktivitas. Mematikan GPU Runtime Colab (runtime.unassign)...", flush=True)
    from google.colab import runtime
    runtime.unassign()